# INF2006 pipeline: fresh-session rerun on EMR

One section per stage, each calling the owned scripts in this repository (no logic lives in the notebook).
Run on a **fresh** EMR cluster that starts from the raw files in S3 only. Every section ends with verification
checks against the committed reference evidence: check, command, expected, actual, evidence location, result.

How to run: `docs/rerun/appendix_a_rerun_guide.md` (Appendix A). Parameters come from environment variables:
`RERUN_ID` (new run id), `S3_ROOT` (`s3://<bucket>/inf2006`), `WITH_GDELT` (1 = full run with the GDELT parse, about 1 hour).

The 2017 test set is **not** evaluated here: it was evaluated once (Task 3, Gate 2). This notebook reloads the saved
model and checks that it reproduces the stored test predictions without retraining.

In [1]:
import os

from rerun_lib import Rerun, get, max_auc_diff, problems_total, q1_paired, q2_paired, q3a_dispersion, repo_root

REPO = repo_root()
os.chdir(REPO)
RERUN_ID = os.environ["RERUN_ID"]
S3_ROOT = os.environ["S3_ROOT"].rstrip("/")
WITH_GDELT = os.environ.get("WITH_GDELT", "1") == "1"
DRY = os.environ.get("DRY_RUN") == "1"

# Reference runs whose committed evidence this rerun must reproduce
REF_T2, REF_MODEL, REF_T4 = "20261105-1430-full-ref", "20261106-1400-full-ref", "20261110-1000-full-ref"
FROZEN = "lr_f1_r001"
PRICES = "/inf2006/raw/prices/plotly-datasets-0c447c4/acquired=2026-11-03"

rr = Rerun(RERUN_ID, dry=DRY)
print("repo", REPO, "| rerun", RERUN_ID, "| with GDELT", WITH_GDELT, "| dry run", DRY)

repo /home/hadoop/quantresearchstuff | rerun 20261116-1000-rerun | with GDELT True | dry run False


## 0. Environment record (software versions, cluster, code version)

In [2]:
rr.sh(
    "date -u; git rev-parse HEAD; git status --short | head -5; python3 --version; hadoop version | head -1; "
    "spark-submit --version 2>&1 | grep -m1 version; hive --version 2>/dev/null | head -1; "
    "grep -io 'emr-[0-9.]*' /mnt/var/lib/info/extraInstanceData.json | head -1; "
    "yarn node -list 2>/dev/null | tail -n +2",
    "environment",
)

[2026-11-15T04:05:50Z] environment: date -u; git rev-parse HEAD; git status --short | head -5; python3 --version; hadoop version | head -1; spark-submit --version 2>&1 | grep -m1 version; hive --version 2>/dev/null | head -1; grep -io 'emr-[0-9.]*' /mnt/var/lib/info/extraInstanceData.json | head -1; yarn node -list 2>/dev/null | tail -n +2


Sun Nov 15 04:05:50 UTC 2026
809a14091c7e9099656b2ae248fc0f389839fd0b
?? evidence/rerun/
Python 3.9.25
Hadoop 3.4.1-amzn-2
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.5-amzn-1
Hive 3.1.3-amzn-19
emr-7.10.0
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
ip-172-31-76-223.ec2.internal:8041	        RUNNING	ip-172-31-76-223.ec2.internal:8042	                           0
ip-172-31-77-23.ec2.internal:8041	        RUNNING	ip-172-31-77-23.ec2.internal:8042	                           0
--- environment: exit 0, 0 min 14 s (full output: evidence/rerun/live_20261116-1000-rerun.log)


'Sun Nov 15 04:05:50 UTC 2026\n809a14091c7e9099656b2ae248fc0f389839fd0b\n?? evidence/rerun/\nPython 3.9.25\nHadoop 3.4.1-amzn-2\n   /___/ .__/\\_,_/_/ /_/\\_\\   version 3.5.5-amzn-1\nHive 3.1.3-amzn-19\nemr-7.10.0\n         Node-Id\t     Node-State\tNode-Http-Address\tNumber-of-Running-Containers\nip-172-31-76-223.ec2.internal:8041\t        RUNNING\tip-172-31-76-223.ec2.internal:8042\t                           0\nip-172-31-77-23.ec2.internal:8041\t        RUNNING\tip-172-31-77-23.ec2.internal:8042\t                           0\n'

## 1. Task 1: stage the raw corpus from S3 into HDFS and verify it
All four sources (price CSV, GDELT, SEC EDGAR, Wikinews) are copied unchanged and checked against the committed manifests.

In [3]:
rr.sh(f"bash corpus/stage_to_hdfs.sh {S3_ROOT}/raw {RERUN_ID}", "task1_stage_and_verify")
inv = rr.load(f"evidence/corpus/inventory_{RERUN_ID}.json") or {}
rr.check(
    "HDFS copies match their manifests (files)",
    "corpus/stage_to_hdfs.sh",
    1047,
    inv.get("files_ok"),
    f"evidence/corpus/inventory_{RERUN_ID}.json",
)
rr.check(
    "Verification problems across all sources",
    "corpus/verify_hdfs.py",
    0,
    problems_total(inv),
    f"evidence/corpus/hdfs_verify_{RERUN_ID}.csv",
)

[2026-11-15T04:06:04Z] task1_stage_and_verify: bash corpus/stage_to_hdfs.sh s3://<bucket>/inf2006/raw 20261116-1000-rerun


    "gdelt_events_v1": {
      "files": 731,
      "bytes": 9611684528,
      "text_lines": 0,
      "problems": 0
    },
    "daily_price_csv": {
      "files": 1,
      "bytes": 29580549,
      "text_lines": 619041,
      "problems": 0
    },
    "sec_edgar_filing": {
      "files": 311,
      "bytes": 2777408405,
      "text_lines": 0,
      "problems": 0
    },
    "wikinews_history": {
      "files": 4,
      "bytes": 946012974,
      "text_lines": 0,
      "problems": 0
    }
  },
  "not_acquired": [],
  "files_ok": 1047,
  "files_checked": 1047
}
wrote evidence/corpus/hdfs_verify_20261116-1000-rerun.csv
--- task1_stage_and_verify: exit 0, 1 min 48 s (full output: evidence/rerun/live_20261116-1000-rerun.log)
PASS  HDFS copies match their manifests (files): expected 1047, actual 1047
PASS  Verification problems across all sources: expected 0, actual 0


True

## 2. Task 2: data pipeline (Streaming audit, PySpark cleaning and features, Hive tables)
`WITH_GDELT=1` also runs the GDELT Streaming job over all 731 archives (about 52 minutes).

In [4]:
rr.sh(f"WITH_GDELT={int(WITH_GDELT)} bash spark/run_task2.sh {RERUN_ID} {PRICES}", "task2_pipeline")
ref = rr.load(f"evidence/pipeline/reconciliation_{REF_T2}.json") or {}
new = rr.load(f"evidence/pipeline/reconciliation_{RERUN_ID}.json") or {}
ev = f"evidence/pipeline/reconciliation_{RERUN_ID}.json"
rr.check("Raw = clean + rejected price rows", "spark/clean_prices.py", True, get(new, "prices", "reconciles"), ev)
rr.check("Clean price rows", "spark/clean_prices.py", 605887, get(new, "prices", "clean_rows"), ev)
for split in ("train", "validation", "test"):
    rr.check(
        f"Eligible stock-weeks, {split}",
        "spark/weekly_features.py",
        get(ref, "features_weekly", split, "eligible"),
        get(new, "features_weekly", split, "eligible"),
        ev,
    )
if WITH_GDELT:
    rr.check(
        "GDELT events read (731 archives)", "streaming/gdelt_daily/run.sh", 144207180, get(new, "gdelt", "lines"), ev
    )
    rr.check("GDELT corrupt archives", "streaming/gdelt_daily/run.sh", 0, get(new, "gdelt", "corrupt_archives"), ev)
    rr.check(
        "Weeks with a pre-start news measure",
        "hive/41_weekly_news.sql",
        97,
        get(new, "weekly_news_weeks_with_news_abn"),
        ev,
    )

[2026-11-15T04:07:52Z] task2_pipeline: WITH_GDELT=1 bash spark/run_task2.sh 20261116-1000-rerun /inf2006/raw/prices/plotly-datasets-0c447c4/acquired=2026-11-03


application_1794715350768_0003
application_1794715350768_0004
application_1794715350768_0005
application_1794715350768_0006
application_1794715350768_0007
application_1794715350768_0008
application_1794715350768_0009
application_1794715350768_0010
application_1794715350768_0011
application_1794715350768_0012
application_1794715350768_0013
application_1794715350768_0014
job_1794715350768_0003
job_1794715350768_0011
--- stage times
stage,start_utc,end_utc,seconds
hive_databases,2026-11-15T04:07:52Z,2026-11-15T04:08:03Z,11
price_audit_streaming,2026-11-15T04:08:03Z,2026-11-15T04:08:43Z,40
clean_prices,2026-11-15T04:08:43Z,2026-11-15T04:09:39Z,56
hive_prices,2026-11-15T04:09:39Z,2026-11-15T04:10:06Z,27
hive_calendar,2026-11-15T04:10:06Z,2026-11-15T04:10:38Z,32
split_screen,2026-11-15T04:10:38Z,2026-11-15T04:11:09Z,31
hive_split_events,2026-11-15T04:11:13Z,2026-11-15T04:11:33Z,20
weekly_features,2026-11-15T04:11:33Z,2026-11-15T04:12:24Z,51
hive_features_weekly,2026-11-15T04:12:24Z,2026-11-1

## 3a. Task 3: rerun model selection on 2016 validation (2017 is not read)
Refits the eight Gate 1 configurations on 2013 to 2015 with seed 2026 and checks that the same configuration is recommended.

In [5]:
rr.sh(f"bash modeling/run_task3.sh validate {RERUN_ID}", "task3_validation")
vref = rr.load(f"evidence/model/validation_{REF_MODEL}.json") or {}
vnew = rr.load(f"evidence/model/validation_{RERUN_ID}.json") or {}
ev = f"evidence/model/validation_{RERUN_ID}.json"
rr.check(
    "Recommended frozen configuration",
    "modeling/run_model.py validate",
    FROZEN,
    get(vnew, "selection", "recommended_frozen_config"),
    ev,
)
rr.check("Test split read during validation", "modeling/run_model.py validate", False, vnew.get("test_split_read"), ev)
rr.check(
    "Max difference in mean weekly AUC, 8 configs vs reference run",
    "modeling/run_model.py validate",
    0.0,
    max_auc_diff(vref, vnew),
    ev,
    tol=1e-6,
)

[2026-11-15T05:06:51Z] task3_validation: bash modeling/run_task3.sh validate 20261116-1000-rerun


        "n_weeks": 52,
        "mean": -0.0012394068745681097,
        "lo": -0.019485062151944003,
        "hi": 0.016589730500243764,
        "sd": 0.06782985560140636,
        "se": 0.009406308553007267
      }
    },
    "rf_f1_d4": {
      "weekly_auc": {
        "n_weeks": 52,
        "mean": 0.4930723193073904,
        "lo": 0.47543778937759323,
        "hi": 0.5104528398090429,
        "sd": 0.06509805627103113,
        "se": 0.009027476146855494
      },
      "rank_ic": {
        "n_weeks": 52,
        "mean": -0.011047686002957195,
        "lo": -0.044127758805746715,
        "hi": 0.02162342676764937,
        "sd": 0.12254082891163483,
        "se": 0.016993355460721518
      },
      "decile_spread_ln_ratio": {
        "n_
wrote /home/hadoop/quantresearchstuff/evidence/model/validation_20261116-1000-rerun.json
RECOMMENDED FROZEN CONFIG: lr_f1_r001 (best rf_f2_d8, 1 SE = 0.0066)
Next: lock Gate 2 in docs/decisions/decision-log.md with the recommended config, push, git pull 

False

## 3b. Task 3: reload the saved model and reproduce the stored test predictions without retraining
The saved PipelineModel comes from this repository (`models/`), the stored predictions and weekly evaluation from S3.

In [6]:
rr.sh(
    f"hdfs dfs -mkdir -p /inf2006/models/{REF_MODEL} && "
    f"hdfs dfs -put -f models/{REF_MODEL}/pipeline_model /inf2006/models/{REF_MODEL}/ && "
    f"s3-dist-cp --src {S3_ROOT}/predictions/{REF_MODEL} --dest hdfs:///inf2006/predictions/{REF_MODEL} && "
    f"s3-dist-cp --src {S3_ROOT}/eval/{REF_MODEL} --dest hdfs:///inf2006/eval/{REF_MODEL}",
    "task3_copy_model_and_predictions",
)
rr.sh(f"spark-submit modeling/run_model.py reload --run-id {REF_MODEL} --root /inf2006 --out {rr.out}", "task3_reload")
rl = rr.load(f"{rr.out}/reload_{REF_MODEL}.json") or {}
ev = f"{rr.out}/reload_{REF_MODEL}.json"
rr.check("Reload check passed (no retraining)", "modeling/run_model.py reload", True, rl.get("passed"), ev)
rr.check("Test predictions compared", "modeling/run_model.py reload", 25531, rl.get("rows_compared"), ev)
rr.check(
    "Max absolute prediction difference", "modeling/run_model.py reload", 0.0, rl.get("max_abs_diff"), ev, tol=1e-9
)

[2026-11-15T05:10:34Z] task3_copy_model_and_predictions: hdfs dfs -mkdir -p /inf2006/models/20261106-1400-full-ref && hdfs dfs -put -f models/20261106-1400-full-ref/pipeline_model /inf2006/models/20261106-1400-full-ref/ && s3-dist-cp --src s3://<bucket>/inf2006/predictions/20261106-1400-full-ref --dest hdfs:///inf2006/predictions/20261106-1400-full-ref && s3-dist-cp --src s3://<bucket>/inf2006/eval/20261106-1400-full-ref --dest hdfs:///inf2006/eval/20261106-1400-full-ref


		Reduce input records=26
		Reduce output records=0
		Spilled Records=52
		Shuffled Maps =3
		Failed Shuffles=0
		Merged Map outputs=3
		GC time elapsed (ms)=189
		CPU time spent (ms)=14610
		Physical memory (bytes) snapshot=1702649856
		Virtual memory (bytes) snapshot=26233221120
		Total committed heap usage (bytes)=1390411776
		Peak Map Physical memory (bytes)=517935104
		Peak Map Virtual memory (bytes)=4529643520
		Peak Reduce Physical memory (bytes)=434270208
		Peak Reduce Virtual memory (bytes)=7242846208
	Shuffle Errors
		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=8463
	File Output Format Counters 
		Bytes Written=0
2026-11-15 05:11:35,998 INFO s3distcp.S3DistCp: Try to recursively delete without throw Exceptionhdfs:/tmp/faf1076f-3d5b-4338-9e2c-6d2c90a6ac12
2026-11-15 05:11:36,010 INFO impl.MetricsSystemImpl: Stopping s3a-file-system metrics system...
2026-11-15 05:11:36,010 INFO impl.MetricsSyste

26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors/json: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors/threadDump: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors/threadDump/json: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors/heapHistogram: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /executors/heapHistogram/json: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /static: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:11:49 INFO ServerInfo: Adding filter to /: org.apache.hadoop.yarn

True

## 4. Task 4: analytics (Hive Q1 and Q3a, PySpark Q2 and Q3b) on this session's features and the stored predictions

In [7]:
rr.sh(f"bash analytics/run_task4.sh {RERUN_ID} /inf2006 {RERUN_ID} {REF_MODEL}", "task4_analytics")
aref = rr.load(f"evidence/analytics/task4_{REF_T4}.json") or {}
anew = rr.load(f"evidence/analytics/task4_{RERUN_ID}.json") or {}
ev = f"evidence/analytics/task4_{RERUN_ID}.json"
rr.check(
    "Q2 paired AUC difference, 2017, all weeks",
    "analytics/run_analytics.py",
    q2_paired(aref),
    q2_paired(anew),
    ev,
    tol=1e-9,
)
rr.check(
    "Q1 busiest minus quietest quintile, 2017",
    "analytics/20_q1_volume_quintiles.sql",
    q1_paired(aref),
    q1_paired(anew),
    ev,
    tol=1e-9,
)
if WITH_GDELT:
    rr.check(
        "Q3a dispersion, high minus low news tercile",
        "analytics/30_q3a_news_market.sql",
        q3a_dispersion(aref),
        q3a_dispersion(anew),
        ev,
        tol=1e-9,
    )

[2026-11-15T05:12:11Z] task4_analytics: bash analytics/run_task4.sh 20261116-1000-rerun /inf2006 20261116-1000-rerun 20261106-1400-full-ref


26/11/15 05:14:30 INFO ServerInfo: Adding filter to /metrics/json: org.apache.hadoop.yarn.server.webproxy.amfilter.AmIpFilter
26/11/15 05:14:30 INFO YarnClientSchedulerBackend: SchedulerBackend is ready for scheduling beginning after reached minRegisteredResourcesRatio: 0.0
26/11/15 05:14:32 WARN HiveConf: HiveConf of name hive.server2.thrift.url does not exist
26/11/15 05:14:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/catalog.py:748: FutureWarning: `dbName` has been deprecated since Spark 3.4 and might be removed in a future version. Use tableExists(`dbName.tableName`) instead.
wrote evidence/analytics/task4_20261116-1000-rerun.json
Q2 validation all       weeks=52 paired model-B3 = -0.0011547174497714568 [-0.026496767340099044, 0.02466499635289778]
Q2 validation vol_low   weeks=52 paired model-B3 = -0.00801

## 5. Verification summary (Appendix A record) and stage times

In [8]:
summary = rr.report()
print(f"\nSaved: {rr.out}/rerun_{RERUN_ID}.json, {rr.out}/checks_{RERUN_ID}.csv, {rr.out}/timings_{RERUN_ID}.csv")


18 of 19 checks passed (rerun 20261116-1000-rerun)

PASS  | HDFS copies match their manifests (files) | expected 1047 | actual 1047
PASS  | Verification problems across all sources | expected 0 | actual 0
PASS  | Raw = clean + rejected price rows | expected True | actual True
PASS  | Clean price rows | expected 605887 | actual 605887
PASS  | Eligible stock-weeks, train | expected 65086 | actual 65086
PASS  | Eligible stock-weeks, validation | expected 25720 | actual 25720
PASS  | Eligible stock-weeks, test | expected 25531 | actual 25531
PASS  | GDELT events read (731 archives) | expected 144207180 | actual 144207180
PASS  | GDELT corrupt archives | expected 0 | actual 0
PASS  | Weeks with a pre-start news measure | expected 97 | actual 97
PASS  | Recommended frozen configuration | expected lr_f1_r001 | actual lr_f1_r001
PASS  | Test split read during validation | expected False | actual False
FAIL  | Max difference in mean weekly AUC, 8 configs vs reference run | expected 0.0 | actua